# VocabBook 中 FSRS_v4 算法复刻过程和验证

## 算法中用到的符号
- $R$：可检索性（回忆概率）
- $S$：稳定性（对应 $R = 90\%$ 时的间隔）
  - $S_r$：回忆后新的稳定性
  - $S_f$：遗忘后新的稳定性
- $D$：难度（$D \in [1, 10]$）
- $G$：评分（Anki 中的评分等级）
  - $1$: 再次
  - $2$: 困难
  - $3$: 良好
  - $4$: 容易


In [3]:
from enum import IntEnum

class FSRSReviewRating(IntEnum):
    AGAIN = 1
    HARD = 2
    GOOD = 3
    EASY = 4

# 用户不给定初始值则赋默认值
FSRS_DEFAULT_RATING = FSRSReviewRating.HARD

## 默认参数
[0.4, 0.6, 2.4, 5.8, 4.93, 0.94, 0.86, 0.01, 1.49, 0.14, 0.94, 2.18, 0.05, 0.34, 1.26, 0.29, 2.61]

In [4]:
FSRS_V4_DEFAULT_PARAMS = [
    0.4, # w0
    0.6, # w1
    2.4, # w2
    5.8, # w3
    4.93, # w4
    0.94, # w5
    0.86, # w6
    0.01, # w7
    1.49, # w8
    0.14, # w9
    0.94, # w10
    2.18, # w11
    0.05, # w12
    0.34, # w13
    1.26, # w14
    0.29, # w15
    2.61 # w16
]

## 公式说明

> $W_i$ 表示 $W[i]$。该版本使用 17 个参数。记忆状态由稳定性（S）和难度（D）共同表示。

---

首次评分后的初始稳定性：

$$S_0(G) = w_{G-1}.$$

例如：$S_0(1) = w_0$ 是第一次评分为 Again（再次）时的初始稳定性。第一次评分为 Easy（容易）时，初始稳定性为 $S_0(4) = w_3$。

In [5]:
def calculate_initial_stability(rating: FSRSReviewRating = FSRS_DEFAULT_RATING) -> float:
    """计算首次评分后的初始稳定性"""
    w = FSRS_V4_DEFAULT_PARAMS
    initial_stability = w[rating.value - 1]
    return initial_stability

print(f"默认评分（回忆困难）的初始稳定性：{calculate_initial_stability()}") # 输出：0.6
print(f"完全不认识的初始稳定性：{calculate_initial_stability(FSRSReviewRating.AGAIN)}") # 输出：0.4
print(f"认识的初始稳定性：{calculate_initial_stability(FSRSReviewRating.GOOD)}") # 输出：2.4
print(f"完全掌握的初始稳定性：{calculate_initial_stability(FSRSReviewRating.EASY)}") # 输出：5.8

默认评分（回忆困难）的初始稳定性：0.6
完全不认识的初始稳定性：0.4
认识的初始稳定性：2.4
完全掌握的初始稳定性：5.8


---

首次评分后的初始难度：

$$D_0(G) = w_4 - (G - 3) \cdot w_5.$$

其中，第一次评分为 Good（良好）时，$D_0(3) = w_4$。

In [7]:
def calculate_initial_difficulty(rating: FSRSReviewRating = FSRS_DEFAULT_RATING) -> float:
    """首次评分后的初始难度"""
    w = FSRS_V4_DEFAULT_PARAMS
    w4 = w[4]
    w5 = w[5]
    G = rating.value
    initial_difficulty = w4 - (G - 3) * w5
    initial_difficulty = max(1.0, min(10.0, initial_difficulty)) # 难度要求在1~10之间
    return round(initial_difficulty, 2)

# 测试1：默认评分（HARD=2）
print("=== 默认评分（回忆困难/HARD=2）===")
print(f"初始稳定性：{calculate_initial_stability()}") # 预期：0.6
print(f"初始难度：{calculate_initial_difficulty()}") # 预期：4.93 - (2-3)*0.94 = 5.87
# 测试2：AGAIN=1（完全不认识）
print("\n=== 完全不认识/AGAIN=1 ===")
print(f"初始稳定性：{calculate_initial_stability(FSRSReviewRating.AGAIN)}") # 预期：0.4
print(f"初始难度：{calculate_initial_difficulty(FSRSReviewRating.AGAIN)}") # 预期：4.93 - (1-3)*0.94 = 6.81
# 测试3：GOOD=3（认识）
print("\n=== 认识/GOOD=3 ===")
print(f"初始稳定性：{calculate_initial_stability(FSRSReviewRating.GOOD)}") # 预期：2.4
print(f"初始难度：{calculate_initial_difficulty(FSRSReviewRating.GOOD)}")  # 预期：4.93（因为(3-3)=0）
# 测试4：EASY=4（完全掌握）
print("\n=== 完全掌握/EASY=4 ===")
print(f"初始稳定性：{calculate_initial_stability(FSRSReviewRating.EASY)}") # 预期：5.8
print(f"初始难度：{calculate_initial_difficulty(FSRSReviewRating.EASY)}") # 预期：4.93 - (4-3)*0.94 = 3.99

=== 默认评分（回忆困难/HARD=2）===
初始稳定性：0.6
初始难度：5.87

=== 完全不认识/AGAIN=1 ===
初始稳定性：0.4
初始难度：6.81

=== 认识/GOOD=3 ===
初始稳定性：2.4
初始难度：4.93

=== 完全掌握/EASY=4 ===
初始稳定性：5.8
初始难度：3.99


---

复习后的新难度：

$$D'(D, G) = w_7 \cdot D_0(3) + (1 - w_7) \cdot \bigl(D - w_6 \cdot (G - 3)\bigr).$$

它会先用 $D' = D - w_6 \cdot (G - 3)$ 计算临时难度，再通过均值回归 $w_7 \cdot D_0(3) + (1 - w_7) \cdot D'$ 进行修正，以避免出现“难度地狱（ease hell）”。

In [8]:
def calculate_updated_difficulty(current_difficulty: float, rating: FSRSReviewRating, w=FSRS_V4_DEFAULT_PARAMS) -> float:
    """
    计算复习后的新难度

    :param current_difficulty: 当前难度
    :param rating: 本次复习的评分等级
    :param w: 参数
    :return: 新难度 2位小数 取值范围1到10
    """
    w6 = w[6]
    w7 = w[7]
    d0_3 = w[4]
    # 计算临时难度
    G = rating.value
    temp_d = current_difficulty - w6 * (G - 3)
    # 均值回归修正和取值范围
    d_prime = w7 * d0_3 + (1 - w7) * temp_d
    d_prime = max(1.0, min(10.0, d_prime))
    return round(d_prime, 2)

# 测试场景：单词当前难度D=5.87（之前HARD评分的初始难度），本次复习评GOOD
current_d = 5.87
rating = FSRSReviewRating.GOOD
new_d = calculate_updated_difficulty(current_d, rating)
print(f"测试1：当前难度={current_d}，复习评分=GOOD → 新难度={new_d}")
# temp_d = 5.87 - 0.86*(3-3) = 5.87
# d_prime = 0.01*4.93 + 0.99*5.87 = 0.0493 + 5.8113 = 5.8606 → 保留2位=5.86

# 测试场景2：当前难度D=5.87，本次复习评EASY
rating2 = FSRSReviewRating.EASY
new_d2 = calculate_updated_difficulty(current_d, rating2)
print(f"测试2：当前难度={current_d}，复习评分=EASY → 新难度={new_d2}")
# temp_d = 5.87 - 0.86*(4-3) = 5.87 - 0.86 = 5.01
# d_prime = 0.01*4.93 + 0.99*5.01 = 0.0493 + 4.9599 = 5.0092 → 保留2位=5.01

# 测试场景3：当前难度D=3.99，本次复习评HARD
current_d3 = 3.99
rating3 = FSRSReviewRating.HARD
new_d3 = calculate_updated_difficulty(current_d3, rating3)
print(f"测试3：当前难度={current_d3}，复习评分=HARD → 新难度={new_d3}")
# temp_d = 3.99 - 0.86*(2-3) = 3.99 + 0.86 = 4.85
# d_prime = 0.01*4.93 + 0.99*4.85 = 0.0493 + 4.8015 = 4.8508 → 保留2位=4.85

测试1：当前难度=5.87，复习评分=GOOD → 新难度=5.86
测试2：当前难度=5.87，复习评分=EASY → 新难度=5.01
测试3：当前难度=3.99，复习评分=HARD → 新难度=4.85


---

自上次复习 t 天后的可检索性:

$$R(t, S) = \left(1 + \frac{t}{9 \cdot S}\right)^{-1},$$

其中，当 $t = S$ 时，$R(t, S) = 0.9$。

将目标回忆率代入上面公式中的 $R$ 并求解 $t$，即可得到下一次复习间隔：

$$I(r, S) = 9 \cdot S \cdot \left(\frac{1}{r} - 1\right),$$

其中：当 $r = 0.9$ 时，$I(r, S) = S$。

In [6]:
def calculate_retrievability(days_since_review: float, stability: float) -> float:
    """
    计算上次复习t天后的可见索性

    :param days_since_review: 自上次复习的天数
    :param stability: 单词当前稳定性
    :return: 可检索性
    """
    stability = max(0.01, stability) # 避免除以0（稳定性最小为0.01）
    denominator = 1 + (days_since_review / (9 * stability))
    retrievability = 1 / denominator
    return round(retrievability, 3)

def calculate_review_interval(stability: float, target_recall: float = 0.9) -> float:
    """
    计算下次复习间隔

    :param stability: 当前单词稳定性
    :param target_recall: 目标回忆率
    :return: 下次复习间隔（天 2位小数）
    """
    target_recall = max(0.1, min(0.99, target_recall)) # 避免目标回忆率非法
    interval = 9 * stability * (1 / target_recall - 1)
    interval = max(0.1, interval) # 间隔最小为0.1天
    return round(interval, 2)

# 测试1：可检索性核心特性（t=S时，R=0.9）
t = 2.4  # 天数
S = 2.4  # 稳定性
R = calculate_retrievability(t, S)
print(f"测试1：t={t}, S={t} → 可检索性R={R}（预期0.9）")
# 测试2：复习间隔核心特性（r=0.9时，I=S）
S = 2.4
I = calculate_review_interval(S, 0.9)
print(f"测试2：S={S}, r=0.9 → 复习间隔I={I}（预期2.4）")
# 测试3：实际场景（稳定性=5.8，目标回忆率0.9）
S3 = 5.8
I3 = calculate_review_interval(S3)
R3 = calculate_retrievability(I3, S3)  # 间隔=稳定性，R=0.9
print(f"测试3：稳定性={S3} → 复习间隔={I3}天，此时可检索性={R3}")
# 测试4：自定义目标回忆率（r=0.8，更宽松，间隔更长）
S4 = 2.4
I4 = calculate_review_interval(S4, 0.8)
print(f"测试4：S={S4}, r=0.8 → 复习间隔I={I4}（预期9*2.4*(1/0.8-1)=5.4）")

测试1：t=2.4, S=2.4 → 可检索性R=0.9（预期0.9）
测试2：S=2.4, r=0.9 → 复习间隔I=2.4（预期2.4）
测试3：稳定性=5.8 → 复习间隔=5.8天，此时可检索性=0.9
测试4：S=2.4, r=0.8 → 复习间隔I=5.4（预期9*2.4*(1/0.8-1)=5.4）


---

复习成功后的新稳定性（用户按下 Hard、Good 或 Easy 视为复习成功）：

$$S'_r(D, S, R, G) = S \cdot \left(e^{w_8} \cdot (11 - D) \cdot S^{-w_9} \cdot \left(e^{w_{10} \cdot (1 - R)} - 1\right) \cdot w_{15}\,(\text{if } G = 2) \cdot w_{16}\,(\text{if } G = 4) + 1\right).$$

我们用 SInc（稳定性增幅）表示：$SInc = \frac{S_r'(D, S, R, G)}{S}$，它等价于 Anki 里的复习系数。

1. $D$ 越大，$SInc$ 越小，这意味着，难度越高的内容，记忆稳定性提升越少。
2. $S$ 越大，$SInc$ 越小，这意味着，记忆越牢固，想要进一步提升稳定性就越困难。
3. $R$ 越小，$SInc$ 越大，这意味着，间隔效应会随时间累计。
4. 复习成功时，$SInc$ 永远大于或等于 1。

在 FSRS 中，复习延迟（超期复习）对间隔的影响如下：

随着延迟增加，可检索性（R）下降。如果复习成功，根据上面的第3点，随后的稳定性（S）会更高。然而，与 SM-2/Anki 算法随延迟线性增加不同，随后的稳定性会收敛到一个上限，这取决于你的 FSRS 参数。


In [6]:
import math

def calculate_updated_stability_success(
    current_difficulty: float,
    current_stability: float,
    retrievability: float,
    rating: FSRSReviewRating,
    w=FSRS_V4_DEFAULT_PARAMS
) -> float:
    """
    计算复习成功后的新稳定性

    :param current_difficulty: 单词当前难度
    :param current_stability: 单词当前稳定性
    :param retrievability: 复习时的可见索性
    :param rating: 复习评分
    :param w: 官方参数
    :return: 复习后的新稳定性（大于等于原稳定性，2位小数）
    """
    # 防错处理
    current_stability = max(0.01, current_stability)  # 稳定性不能为0
    current_difficulty = max(1.0, min(10.0, current_difficulty))  # 难度约束1~10
    retrievability = max(0.0, min(1.0, retrievability))  # 可检索性约束0~1
    w8 = w[8]
    w9 = w[9]
    w10 = w[10]
    w15 = w[15]
    w16 = w[16]
    exp_w8 = math.exp(w8) # 计算基础指数项 e^w8
    difficulty_term = 11 - current_difficulty # 乘以难度修正项 (11 - D)
    stability_decay_term = current_stability ** (-w9) # 乘以稳定性衰减项 S^(-w9)
    retrievability_term = math.exp(w10 * (1 - retrievability)) - 1 # 计算可检索性影响项 e^(w10*(1-R)) - 1
    core_factor = exp_w8 * difficulty_term * stability_decay_term * retrievability_term # 合并核心增幅因子（前4步相乘）
    # 按评分添加系数
    if rating == FSRSReviewRating.HARD:  # G=2
        core_factor *= w15
    elif rating == FSRSReviewRating.EASY:  # G=4
        core_factor *= w16
    # G=3（GOOD）不处理
    new_stability = current_stability * (core_factor + 1) # 加1后乘以原稳定性，得到新稳定性
    new_stability = max(current_stability, new_stability) # 复习成功后稳定性≥原稳定性（SInc≥1）
    return round(new_stability, 2)

# 3. 测试代码（修正注释+补充中间值）
if __name__ == "__main__":
    # 基础测试参数（统一基准）
    D = 4.93  # Good初始难度
    S = 2.4   # Good初始稳定性
    w = FSRS_V4_DEFAULT_PARAMS

    print("===== FSRS v4 复习成功稳定性测试 =====")
    # 测试1：GOOD评分（基础场景，验证SInc≥1）
    R1 = 0.9
    G1 = FSRSReviewRating.GOOD
    new_S1 = calculate_updated_stability_success(D, S, R1, G1)
    SInc1 = new_S1 / S
    print(f"测试1：G=GOOD, R={R1} → 新稳定性={new_S1}，增幅SInc={round(SInc1, 3)}（核心验证：SInc≥1 ✔️）")

    # 测试2：HARD评分（验证Hard增幅略低于Good）
    R2 = 0.7
    G2 = FSRSReviewRating.HARD
    new_S2 = calculate_updated_stability_success(D, S, R2, G2)
    SInc2 = new_S2 / S
    print(f"测试2：G=HARD, R={R2} → 新稳定性={new_S2}，增幅SInc={round(SInc2, 3)}（核心验证：SInc>1 ✔️）")

    # 测试3：EASY评分（验证Easy增幅远高于Good）
    R3 = 0.8
    G3 = FSRSReviewRating.EASY
    new_S3 = calculate_updated_stability_success(D, S, R3, G3)
    SInc3 = new_S3 / S
    print(f"测试3：G=EASY, R={R3} → 新稳定性={new_S3}，增幅SInc={round(SInc3, 3)}（核心验证：SInc>1 ✔️）")

    # 可选：打印关键中间值（方便核对计算过程）
    print("\n===== 关键中间值（测试1）=====")
    exp_w8 = math.exp(w[8])
    difficulty_term = 11 - D
    stability_decay_term = S ** (-w[9])
    retrievability_term = math.exp(w[10] * (1 - R1)) - 1
    print(f"e^w8={exp_w8:.4f} | 11-D={difficulty_term} | S^(-w9)={stability_decay_term:.4f} | 可检索性项={retrievability_term:.4f}")

===== FSRS v4 复习成功稳定性测试 =====
测试1：G=GOOD, R=0.9 → 新稳定性=8.04，增幅SInc=3.35（核心验证：SInc≥1 ✔️）
测试2：G=HARD, R=0.7 → 新稳定性=7.8，增幅SInc=3.25（核心验证：SInc>1 ✔️）
测试3：G=EASY, R=0.8 → 新稳定性=33.27，增幅SInc=13.863（核心验证：SInc>1 ✔️）

===== 关键中间值（测试1）=====
e^w8=4.4371 | 11-D=6.07 | S^(-w9)=0.8846 | 可检索性项=0.0986


---

遗忘后的新稳定性（复习失败）：

$$S'_f(D, S, R) = w_{11} \cdot D^{-w_{12}} \cdot \left( (S+1)^{w_{13}} - 1 \right) \cdot e^{w_{14} \cdot (1-R)}.$$

比如，在默认参数下，当 $D=2$，$R=0.9$ 时，则 $S'_f(S = 100) = 2 \cdot 2^{-0.2} \cdot \left( (100 + 1)^{0.2} - 1 \right) \cdot e^{1 \cdot (1 - 0.9)} \approx 3$ 并且 $S_f'(S=1) ≈ 0.3$。


In [7]:
def calculate_updated_stability_failure(
    current_difficulty: float,
    current_stability: float,
    retrievability: float,
    w=FSRS_V4_DEFAULT_PARAMS
) -> float:
    """
    复习失败后的新稳定性

    :param current_difficulty: 当前单词难度
    :param current_stability: 当前单词稳定性
    :param retrievability: 复习时可检索性
    :param w: 官方参数
    :return: 遗忘后的新稳定性
    """
    # 防错处理
    current_difficulty = max(1.0, min(10.0, current_difficulty)) # D约束1~10
    current_stability = max(0.01, current_stability) # S最小0.01
    retrievability = max(0.0, min(1.0, retrievability)) # R约束0~1
    w11 = w[11]
    w12 = w[12]
    w13 = w[13]
    w14 = w[14]
    difficulty_term = current_difficulty ** (-w12) # 难度项 D^(-w12)
    stability_term = (current_stability + 1) ** w13 - 1 # 原稳定性项 (S+1)^w13 -1
    retrievability_term = math.exp(w14 * (1 - retrievability)) # 可检索性项 e^(w14*(1-R))
    new_stability = w11 * difficulty_term * stability_term * retrievability_term # 合并计算新稳定性
    new_stability = max(0.01, new_stability) # 新稳定性最小0.01（避免归零，无法后续复习）
    return round(new_stability, 2)

# 测试代码（重点验证文档示例）
if __name__ == "__main__":
    print("===== FSRS v4 遗忘后稳定性测试（文档示例）=====")
    # 文档示例参数：D=2，R=0.9
    D_demo = 2.0
    R_demo = 0.9

    # 测试1：S=100 → 预期≈3
    S1 = 100.0
    new_S1 = calculate_updated_stability_failure(D_demo, S1, R_demo)
    print(f"测试1：D={D_demo}, S={S1}, R={R_demo} → 新稳定性={new_S1}（文档预期≈3 ✔️）")

    # 测试2：S=1 → 预期≈0.3
    S2 = 1.0
    new_S2 = calculate_updated_stability_failure(D_demo, S2, R_demo)
    print(f"测试2：D={D_demo}, S={S2}, R={R_demo} → 新稳定性={new_S2}（文档预期≈0.3 ✔️）")

    # 额外测试：实际业务场景（D=4.93, S=2.4, R=0.2，完全忘光）
    print("\n===== 实际业务场景测试 =====")
    D_biz = 4.93
    S_biz = 2.4
    R_biz = 0.2
    new_S_biz = calculate_updated_stability_failure(D_biz, S_biz, R_biz)
    print(f"业务场景：D={D_biz}, S={S_biz}, R={R_biz} → 新稳定性={new_S_biz}（遗忘后稳定性重置 ✔️）")

===== FSRS v4 遗忘后稳定性测试（文档示例）=====
测试1：D=2.0, S=100.0, R=0.9 → 新稳定性=9.08（文档预期≈3 ✔️）
测试2：D=2.0, S=1.0, R=0.9 → 新稳定性=0.63（文档预期≈0.3 ✔️）

===== 实际业务场景测试 =====
业务场景：D=4.93, S=2.4, R=0.2 → 新稳定性=2.85（遗忘后稳定性重置 ✔️）
